In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

In [2]:
X_train_class = pd.read_csv('../data/processed/X_train_class.csv')
X_test_class = pd.read_csv('../data/processed/X_test_class.csv')
y_train_class = pd.read_csv('../data/processed/y_train_class.csv').squeeze()
y_test_class = pd.read_csv('../data/processed/y_test_class.csv').squeeze()

SEED = 42

In [3]:
tree_profunda = DecisionTreeClassifier(criterion='gini', max_depth=None, random_state=SEED)
tree_profunda.fit(X_train_class, y_train_class)

acc_treino_profunda = accuracy_score(y_train_class, tree_profunda.predict(X_train_class))
acc_teste_profunda = accuracy_score(y_test_class, tree_profunda.predict(X_test_class))

print("=== 1. Árvore Profunda (Demonstração de Overfitting) ===")
print(f"Acurácia Treino: {acc_treino_profunda:.4f} | Acurácia Teste: {acc_teste_profunda:.4f}")
print(f"Gap de Generalização: {acc_treino_profunda - acc_teste_profunda:.4f}\n")

=== 1. Árvore Profunda (Demonstração de Overfitting) ===
Acurácia Treino: 1.0000 | Acurácia Teste: 0.6587
Gap de Generalização: 0.3413



In [4]:
tree_gini = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=SEED)
tree_gini.fit(X_train_class, y_train_class)

# Árvore com Entropia
tree_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=5, random_state=SEED)
tree_entropy.fit(X_train_class, y_train_class)

print("=== 2. Comparação de Critérios (max_depth=5) ===")
print(f"Acurácia Teste (Gini): {accuracy_score(y_test_class, tree_gini.predict(X_test_class)):.4f}")
print(f"Acurácia Teste (Entropia): {accuracy_score(y_test_class, tree_entropy.predict(X_test_class)):.4f}\n")

=== 2. Comparação de Critérios (max_depth=5) ===
Acurácia Teste (Gini): 0.7143
Acurácia Teste (Entropia): 0.7063



In [5]:
parametros = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 4, 5, 6],
    'min_samples_split': [2, 4, 8, 16, 32],
    'min_samples_leaf': [1, 2, 4, 8, 16]
}

In [6]:
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=SEED),
    param_grid=parametros,
    cv=5,  # Exigência do Professor (K-fold com K=5)
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_class, y_train_class)

# A melhor árvore encontrada
melhor_arvore = grid_search.best_estimator_

print("=== 3. Melhor Árvore Otimizada ===")
print(f"Melhores hiperparâmetros: {grid_search.best_params_}")
print(f"Acurácia Validação Cruzada (Treino): {grid_search.best_score_:.4f}")
print(f"Acurácia Teste Final: {accuracy_score(y_test_class, melhor_arvore.predict(X_test_class)):.4f}\n")

=== 3. Melhor Árvore Otimizada ===
Melhores hiperparâmetros: {'criterion': 'entropy', 'max_depth': 2, 'min_samples_leaf': 1, 'min_samples_split': 2}
Acurácia Validação Cruzada (Treino): 0.7740
Acurácia Teste Final: 0.7540



In [7]:
print("=== 4. Estatísticas da Árvore Otimizada ===")
print(f" - Profundidade (Níveis): {melhor_arvore.get_depth()}")
print(f" - Quantidade de Folhas: {melhor_arvore.get_n_leaves()}")
print(f" - Quantidade de Nós (estimativa): {melhor_arvore.tree_.node_count}\n")

print("=== 5. As três primeiras ramificações (Para comparação com o Sistema Especialista) ===")
regras_arvore = export_text(melhor_arvore, feature_names=list(X_train_class.columns), max_depth=3)
print(regras_arvore)

=== 4. Estatísticas da Árvore Otimizada ===
 - Profundidade (Níveis): 2
 - Quantidade de Folhas: 4
 - Quantidade de Nós (estimativa): 7

=== 5. As três primeiras ramificações (Para comparação com o Sistema Especialista) ===
|--- prior_gpa <= 0.45
|   |--- prior_gpa <= -0.70
|   |   |--- class: 0
|   |--- prior_gpa >  -0.70
|   |   |--- class: 0
|--- prior_gpa >  0.45
|   |--- prior_gpa <= 0.76
|   |   |--- class: 1
|   |--- prior_gpa >  0.76
|   |   |--- class: 1

